# Mixed elasticity: H(div) stress and weak symmetry

This notebook uses the native BDM2 / discontinuous-P1 displacement / discontinuous-P1 rotation formulation. The three local rigid motions are exact. The skeleton is negative Cauchy traction; rotation approximates `(du_x/dy-du_y/dx)/2`.

The spaces and default interior/exterior traction choices follow the [2021 MHM weak-symmetry elasticity formulation](https://doi.org/10.1051/m2an/2021013). The numerical records below verify analytical data on the stated pyMHM meshes; they do not reproduce the paper's table ordinates.


In [ ]:
import json
from pathlib import Path
import numpy as np
from IPython.display import Image, display
from pymhm import TriangleMesh, solve_elasticity_mixed

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists())


## Non-affine solenoidal patch

For `u=(x²,-2xy)`, `mu=1`, and any finite nonnegative `lambda`, the exact stress is `[[4x,-2y],[-2y,-4x]]`, force is `(-2,0)`, and rotation is `q=y`. The discrete displacement is its cellwise P1 L2 projection; its L2 error is therefore nonzero.


In [ ]:
def displacement(x):
    """Quadratic divergence-free displacement."""
    return np.column_stack((x[:, 0]**2, -2*x[:, 0]*x[:, 1]))


def stress(x):
    """Analytical Cauchy stress for mu=1."""
    value = np.zeros((len(x), 2, 2))
    value[:, 0, 0], value[:, 1, 1] = 4*x[:, 0], -4*x[:, 0]
    value[:, 0, 1] = value[:, 1, 0] = -2*x[:, 1]
    return value

solution = solve_elasticity_mixed(
    TriangleMesh.unit_square(2), lame_lambda=1e6, lame_mu=1,
    source=(-2, 0), dirichlet=displacement,
)
assert solution.stress_l2_error(stress) < 1e-8
assert solution.rotation_l2_error(lambda x: x[:, 1]) < 2e-12
assert np.max(np.abs(solution.equilibrium_residuals())) < 3e-12
{"displacement_L2": solution.l2_error(displacement),
 "stress_L2": solution.stress_l2_error(stress),
 "force_and_moment_max": float(np.max(np.abs(solution.equilibrium_residuals())))}


## Five meshes, seven Lamé ratios, and physical moments

The archived analytical study uses diagonal unit-square macrotriangles, local refinement two, interior P1 traction, and exterior P2 on each fine boundary edge. Errors are integrated with order-10 quadrature. Weak symmetry refers to P1 moments; the raw tensor need not be symmetric pointwise. The native integration suite separately compares all fields with independently assembled DOLFINx/UFL and Basix BDM2/P1/P1.


In [ ]:
records = json.loads((ROOT / "examples/results/mixed-elasticity.json").read_text())
records["convergence"]


In [ ]:
display(Image(filename=str(ROOT / "docs/figures/mixed-elasticity/convergence-and-incompressibility.png")))
display(Image(filename=str(ROOT / "docs/figures/mixed-elasticity/smooth-fields.png")))


## Oscillatory modulus analytical case

The 2021 data use `E=100[1+0.3 sin(10π(x−0.5))cos(10πy)]`, `nu=0.3`, and displacement `(x²y² cos(6πx)sin(7πy)/27, exp(y)sin(4πx)/5)`. The prescribed boundary data are the actual nonhomogeneous trace. Force includes the gradients of E and is checked independently by complex-step stress differentiation in the public analytical gallery example. These results use 512 macrotriangles and are not an assertion that Table 3 has been reproduced.


In [ ]:
records["oscillatory_2021_analytical_data"]


In [ ]:
display(Image(filename=str(ROOT / "docs/figures/mixed-elasticity/oscillatory-fields.png")))


## Limits

The finite-coefficient sweep reaches `lambda/mu=1e6` (`nu≈0.4999995`). At much larger ratios, floating-point error in mean hydrostatic stress can grow; infinite lambda is not supported. Uniform behavior for this stable family does not establish robustness for arbitrary coefficient contrast or arbitrary skeleton choices. Stress divergence errors and integrated equilibrium defects measure different quantities.


## Independent assembly of the published oscillatory case

DOLFINx/UFL/Basix assembles the same variable modulus, source and nonzero displacement trace. A native normal-moment restriction preserves the declared macro traction space, including the enriched local families. The compared quantities are physical stress, displacement and independent rotation. Native code agreement and analytical convergence do not resolve the article's unreported mesh convention; the historical table ordinates remain distinct.


In [ ]:
native = json.loads((ROOT / "examples/results/mixed-elasticity/native-discrete-verification.json").read_text())
for row in native["rows"]:
    norms = row["comparisons"][-1]
    print(row["case"], {field: norms[field]["relative_difference"] for field in ("stress", "displacement", "rotation")})
display(Image(filename=str(ROOT / "docs/figures/mixed-elasticity/native-displacement-rotation.png")))
display(Image(filename=str(ROOT / "docs/figures/mixed-elasticity/native-stress.png")))
